# *Missa Veni sponsa Christi* — Cadenze, Omoritmia e Presentation Types

Questo notebook analizza tre aspetti del mottetto *Veni sponsa Christi* e delle cinque parti della *Missa Veni sponsa Christi* di Giovanni Pierluigi da Palestrina:

- **Cadenze** — riconoscimento automatico delle cadenze con indicazione del tipo e della sonorità di approdo.
- **Omoritmia** — individuazione dei passaggi in cui più voci procedono omoritmicamente.
- **Presentation Types** — classificazione degli episodi imitativi.

Le composizioni sono caricate direttamente dal [CRIM Project](https://crimproject.org/).
Le partiture in formato PDF sono disponibili sul sito del progetto:<br>
[Veni sponsa Christi](https://crimproject.org/pieces/CRIM_Model_0019/)<br>
[Missa Veni sponsa Christi](https://crimproject.org/masses/CRIM_Mass_0019/)

| Codice | Composizione |
|--------|-------------|
| `CRIM_Model_0019` | *Veni sponsa Christi* — mottetto (modello) |
| `CRIM_Mass_0019_1` | *Missa Veni sponsa Christi*: Kyrie |
| `CRIM_Mass_0019_2` | *Missa Veni sponsa Christi*: Gloria |
| `CRIM_Mass_0019_3` | *Missa Veni sponsa Christi*: Credo |
| `CRIM_Mass_0019_4` | *Missa Veni sponsa Christi*: Sanctus |
| `CRIM_Mass_0019_5` | *Missa Veni sponsa Christi*: Agnus Dei |

**Come usare il notebook:** esegui le celle nell'ordine seguendo le istruzioni di ogni sezione. Per visualizzare gli esempi musicali in partitura è necessaria una connessione a internet.

---

## 1. Avvio

Importa le librerie necessarie. Esegui questa cella prima di qualsiasi altra.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from crim_intervals import importScore
from IPython.display import HTML, display
import pandas as pd

print('Ok')

## 2. Lista delle composizioni

La lista `piece_list` contiene i codici CRIM delle composizioni da analizzare. Per includere o escludere una parte della messa è sufficiente aggiungere o rimuovere il codice corrispondente dalla lista. È possibile duplicare e modificare questa cella per creare una lista personalizzata (per esempio un altro gruppo modello+messa selezionato dal `CRIM Corpus`).

In [ ]:
piece_list = [
    'CRIM_Model_0019',   # mottetto (modello)
    'CRIM_Mass_0019_1',  # Kyrie
    'CRIM_Mass_0019_2',  # Gloria
    'CRIM_Mass_0019_3',  # Credo
    'CRIM_Mass_0019_4',  # Sanctus
    'CRIM_Mass_0019_5',  # Agnus Dei
]
print(f'{len(piece_list)} composizioni in lista')

## 3. Cadenze

`piece.cadences()` individua le cadenze sulla base di pattern contrappuntistici riconoscibili. Per ogni cadenza vengono registrati: il **tipo** (`CadType`), le sigle delle *clausulae* coinvolte nella cadenza (`CVFs` = Candential Voice Functions), la **sonorità di approdo** (`Tone`), la posizione nella composizione (`Measure`, `Beat`, `Progress`) e altri parametri. Per la descrizione delle tipologie delle cadenze si rimanda al [CRIM Musical Types Thesaurus](https://github.com/CRIM-Project/CRIM-online/blob/master/CRIM_Musical_Types_Thesaurus.md#cadences).

`piece.linkExamples()` genera, per ogni riga della tabella, un collegamento ipertestuale che apre l'esempio direttamente in partitura nel CRIM Project, con le note interessate evidenziate in rosso. Funziona soltanto con file caricati da URL pubblici.

### 3a. Cadenze in una singola composizione

Carica una composizione e calcola le sue cadenze. Modifica `url` per analizzare una parte diversa della messa o un'altra composizione.

In [ ]:
url = 'https://crimproject.org/mei/CRIM_Model_0019.mei'
piece = importScore(url)
print(piece.metadata)
piece.cadences()

In [ ]:
cad = piece.cadences()
link_cad = piece.linkExamples(df=cad, piece_url=url)
link_cad

Per un approfondimento sulle Cadential Voice Functions (`CVFs`) esegui la cella sottostante.

In [ ]:
print(piece.cvfs.__doc__)

### 3b. Cadenze nel corpus

La cella seguente carica ogni composizione della lista e visualizza i link alle cadenze individuate.

In [ ]:
for mei_file in piece_list:
    url = 'https://crimproject.org/mei/' + mei_file + '.mei'
    piece = importScore(url)
    if piece is None:
        print(f'Errore nel caricamento: {mei_file}')
        continue
    print(piece.metadata)
    cad = piece.cadences()
    piece.linkExamples(df=cad, piece_url=url)

## 4. Omoritmia

`piece.homorhythm()` individua i passaggi in cui più voci procedono con lo stesso ritmo (*omoritmia*). Per la definizione si rimanda al [CRIM Musical Types Thesaurus](https://github.com/CRIM-Project/CRIM-online/blob/master/CRIM_Musical_Types_Thesaurus.md#homorhythm). Parametri principali:

- **`ngram_length`** — numero minimo di sillabe consecutive intonate omoritmicamente.
- **`full_hr`** — se `True`, vengono inclusi solo i passaggi in cui *tutte* le voci attive procedono omoritmicamente; se `False`, è sufficiente che lo facciano due voci qualsiasi.

### 4a. Omoritmia in una singola composizione

Modifica `ngram_length` per variare la soglia minima di durata dei passaggi restituiti. Modifica `url` per analizzare una parte diversa della messa o un'altra composizione.

In [ ]:
url = 'https://crimproject.org/mei/CRIM_Mass_0019_3.mei'
piece = importScore(url)
print(piece.metadata)
piece.homorhythm(ngram_length=4, full_hr=True)

In [ ]:
hr = piece.homorhythm(ngram_length=4, full_hr=True)
link_hr = piece.linkExamples(df=hr, piece_url=url)
link_hr

### 4b. Omoritmia nel corpus

Vengono analizzate tutte le composizioni in `piece_list`. intervieni sui parametri `ngram_length` (indica un numero) e `full_hr` (indica **True** o **False**) per affinare la ricerca. Se in una composizione non vengono trovati passaggi omoritmici con i parametri impostati, la composizione viene saltata con un messaggio.

In [ ]:
ngram_length=5 # ← modifica questo parametro
full_hr=True # ← modifica questo parametro

for mei_file in piece_list:
    url = 'https://crimproject.org/mei/' + mei_file + '.mei'
    piece = importScore(url)
    if piece is None:
        print(f'Errore nel caricamento: {mei_file}')
        continue
    print(piece.metadata)
    hr = piece.homorhythm(ngram_length=ngram_length, full_hr=full_hr)
    if hr is None or hr.empty:
        print(f"Nessun passaggio omoritmico di almeno {ngram_length} sillabe trovato in: {piece.metadata.get('title', mei_file)}")
        continue
    piece.linkExamples(df=hr, piece_url=url)

## 5. Presentation Types (impianti imitativi)

`piece.presentationTypes()` classifica gli episodi imitativi della composizione in base al tipo di procedimento imitativo. Parametri principali:

- **`melodic_ngram_length`** — lunghezza del soggetto espressa come numero di **intervalli** (es. `4` = quattro intervalli = cinque note).
- **`limit_to_entries`** — se `True`, vengono considerate solo le occorrenze che costituiscono un'*entrata*, ossia che compaiono dopo una pausa o all'inizio della composizione.
- **`include_hidden_types`** — se `True`, include anche le combinazioni interne delle entrate che formano un presentation type per individuare casi particolari (**PEN = Periodic entries**, **ID = Imitative Duo**) che potrebbero essere nascosti all'interno di una **FUGA** con molte entrate.
- **`combine_unisons`** — se `True`, i ribattuti vengono raggruppati in un'unica nota.

Per la descrizione degli impianti imitativi, o tipi di presentazione (`Presentation_Type`), si rimanda al [CRIM Musical Types Thesaurus](https://github.com/CRIM-Project/CRIM-online/blob/master/CRIM_Musical_Types_Thesaurus.md#schuberts-presentation-types-in-brief).

### 5a. Presentation types in una singola composizione

Ogni riga della tabella corrisponde a un episodio imitativo; la colonna `Presentation_Type` indica il tipo. Modifica `url` per analizzare una parte diversa della messa o un'altra composizione.

In [ ]:
url = 'https://crimproject.org/mei/CRIM_Model_0019.mei'
piece = importScore(url)
print(piece.metadata)
piece.presentationTypes(melodic_ngram_length=4, limit_to_entries=True, include_hidden_types=False, combine_unisons=True)

In [ ]:
pt = piece.presentationTypes(melodic_ngram_length=4, limit_to_entries=True, include_hidden_types=False, combine_unisons=True)
link_pt = piece.linkExamples(df=pt, piece_url=url)
link_pt

### 5b. Presentation types nel corpus

In caso di errore nel caricamento di una composizione (es. per problemi di connessione), l'analisi prosegue automaticamente con la composizione successiva.

In [ ]:
for mei_file in piece_list:
    url = 'https://crimproject.org/mei/' + mei_file + '.mei'
    piece = importScore(url)
    if piece is None:
        print(f'Errore nel caricamento: {mei_file}')
        continue
    print(piece.metadata)
    pt = piece.presentationTypes(melodic_ngram_length=4, limit_to_entries=True, include_hidden_types=False, combine_unisons=True)
    if pt is None or pt.empty:
        print(f"Nessun tipo di presentazione trovato in: {piece.metadata.get('title', mei_file)}")
        continue
    piece.linkExamples(df=pt, piece_url=url)